In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'


In [2]:
data = pd.read_csv(url)
data.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
data.shape

(5000, 9)

### Data preparation

Check if the missing values are presented in the features.
    
If there are missing values:
    
For categorical features, replace them with 'NA'

For numerical features, replace with with 0.0

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


In [5]:
data.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [6]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income','number_of_courses_viewed', 'interaction_count','lead_score']

In [7]:
data.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

### Question 1: What is the most frequent observation (mode) for the column industry?

In [8]:
data.industry.mode()

0    technology
Name: industry, dtype: object

In [9]:
data[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.137542,0.106375,0.231396
number_of_courses_viewed,0.137542,1.000000,0.721609,0.769352
interaction_count,0.106375,0.721609,1.000000,0.928712
lead_score,0.231396,0.769352,0.928712,1.000000


In [10]:
##filling missing value
df = data.copy()

df[numerical] = df[numerical].fillna(0.0)
df[categorical] = df[categorical].fillna('NA')

In [11]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [12]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [13]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [14]:
y_train = df_train.converted
y_val = df_val.converted
y_test = df_test.converted

del df_train['converted']
del df_val['converted']
del df_test['converted']

### Question 3: Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only. Round the scores to 2 decimals using round(score, 2).
Which of these variables has the biggest mutual information sco

In [15]:
df_full_train.converted.value_counts(normalize=True)

converted
1    0.57075
0    0.42925
Name: proportion, dtype: float64

In [16]:
df_full_train[categorical].nunique()

lead_source          6
industry             7
employment_status    5
location             6
dtype: int64

In [17]:
from sklearn.metrics import mutual_info_score
mutual_info_score(df_full_train.converted, df_full_train.lead_source)

0.0347609795100606

In [18]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)
mi = df_full_train[categorical].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

lead_source          0.034761
employment_status    0.020313
industry             0.002898
location             0.001050
dtype: float64

### Question 4: Logistic regression.

In [19]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression


features = categorical + numerical

def train_evaluate(features):
    dv = DictVectorizer(sparse=False)

    train_dict = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    convert_decision = y_pred >= 0.5

    accuracy = (y_val == convert_decision).mean()

    return accuracy

In [23]:
general_accuracy = train_evaluate(features)

print('General accuracy:', round(general_accuracy, 3))

General accuracy: 0.645


### Question 5: Find the least useful feature using the feature elimination technique.

Train a model using the same features and parameters as in Q4 (without rounding).

Now exclude each feature from this set and train a model without it. Record the accuracy for each model.

For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

In [21]:
results = []

for feature in features:
    features_without = [f for f in features if f != feature]

    accuracy = train_evaluate(features_without)

    results.append({
        'removed_feature': feature,
        'accuracy': accuracy,
        'difference': accuracy - general_accuracy
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('difference', ascending=False)

print(results_df.round(4))

            removed_feature  accuracy  difference
4             annual_income     0.724       0.079
1                  industry     0.645       0.000
3                  location     0.645       0.000
2         employment_status     0.645       0.000
7                lead_score     0.644      -0.001
5  number_of_courses_viewed     0.643      -0.002
0               lead_source     0.642      -0.003
6         interaction_count     0.601      -0.044


The smallest absolute difference is -0.002, so 'number_of_courses_viewed'.

### Question 6: Train a regularized logistic regression.
Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
Train models using all the features as in Q4.
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these C leads to the best accuracy on the validation set?

In [22]:

C_values = [0.000001, 0.00001, 0.0001, 0.001]

dv = DictVectorizer(sparse=False)

train_dict = df_train[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[features].to_dict(orient='records')
X_val = dv.transform(val_dict)

for C in C_values:
    model = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    accuracy = ((y_pred >= 0.5) == y_val).mean()

    print(f'C={C}: accuracy={accuracy:.3f}')

C=1e-06: accuracy=0.598
C=1e-05: accuracy=0.598
C=0.0001: accuracy=0.613
C=0.001: accuracy=0.645
